# [9665] Exercise : Locality-Sensitive Hashing - Solution
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/TED_talks.csv

## Exercise Requirements
* Load data into dataframe
* Perform text preprocessing (follow steps carefully)
* Prepare data for model training and testing
* Part 1:
  * Create functions:
    - to generate MinHash Forest (shingle based on word boundary)
    - to query MinHash Forest (shingle based on word boundary)
  * Execute functions to create forest (shingle based on word boundary)
  * Query Forest (shingle based on word boundary) to make recommendations for 2 randomly selected TED talks
* Part 2:
  * Create functions:
    - to create shingles
    - to generate MinHash Forest (fixed shingle size based on X characters)
    - to query MinHash Forest (fixed shingle size based on X characters)
  * Execute functions to create forest (fixed shingle size based on X characters)
  * Query Forest (fixed shingle size based on X characters) to make recommendations for 2 randomly selected TED talks

In [1]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 02/17/25 12:36:15


### Import libraries

In [2]:
%%time

! pip install datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 kB 1.7 MB/s eta 0:00:00
CPU times: user 57.8 ms, sys: 16.3 ms, total: 74.1 ms
Wall time: 7.08 s


In [3]:
import numpy as np
import pandas as pd
import time
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from datasketch import MinHash
from datasketch import MinHashLSHForest

In [4]:
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Load data

In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/TED_talks.csv')
df.shape

(3923, 9)

### Examine data

In [6]:
df.head()

,video_link,thumbnail_link,duration,title,views,likes,comments,date,description
0,https://www.youtube.com//watch?v=B5smctuV7-Q,https://i.ytimg.com/vi/B5smctuV7-Q/hqdefault.jpg,"14 minutes, 48 seconds",We Can Make COVID-19 the Last Pandemic | Bill ...,"208,480",Like,NaN,22 Apr 2022,"Building a pandemic-free future won’t be easy,..."
1,https://www.youtube.com//watch?v=FrqBWQ-mVEc,https://i.ytimg.com/vi/FrqBWQ-mVEc/hqdefault.jpg,"9 minutes, 53 seconds",The Future Will Be Shaped by Optimists | Kevin...,"51,273",1.9K,262,21 Apr 2022,"""Every great and difficult thing has required ..."
2,https://www.youtube.com//watch?v=iIne-UO7wUo,https://i.ytimg.com/vi/iIne-UO7wUo/hqdefault.jpg,"9 minutes, 18 seconds",An Olympic Champion’s Unwavering Advocacy for ...,"25,597",Like,53,20 Apr 2022,Getting pregnant as a track and field athlete ...
3,https://www.youtube.com//watch?v=5T2VRY0LECc,https://i.ytimg.com/vi/5T2VRY0LECc/hqdefault.jpg,"6 minutes, 56 seconds",The African Swamp Protecting Earth's Environme...,"21,430",633,40,20 Apr 2022,The peatlands of Africa's Congo Basin are a va...
4,https://www.youtube.com//watch?v=YRvf00NooN8,https://i.ytimg.com/vi/YRvf00NooN8/hqdefault.jpg,"1 hour, 6 minutes, 25 seconds",Elon Musk: A future worth getting excited abou...,"3,609,893",95K,"8,757",18 Apr 2022,What's on Elon Musk's mind? In this exclusive ...


In [7]:
pd.set_option('display.max_colwidth', None)

In [8]:
df['description'].head()

,description
0,"Building a pandemic-free future won’t be easy, but Bill Gates believes that we have the tools and strategies to make it possible -- now we just have to fund them. In this forward-looking talk, he proposes a multi-specialty Global Epidemic Response and Mobilization (GERM) team that would detect potential outbreaks and stop them from becoming pandemics. By investing in disease monitoring, research and development as well as improved health systems, Gates believes we can “create a world where everyone has a chance to live a healthy and productive life -- a life free from the fear of the next COVID-19.”\n\nIf you love watching TED Talks like this one, become a TED Member to support our mission of spreading ideas: http://ted.com/membership\n\nFollow TED! \nTwitter: http://twitter.com/TEDTalks\nInstagram: https://www.instagram.com/ted\nFacebook: http://facebook.com/TED\nLinkedIn: https://www.linkedin.com/company/ted-...\nTikTok: https://www.tiktok.com/@tedtoks\n\nThe TED Talks channel features talks, performances and original series from the world's leading thinkers and doers. Subscribe to our channel for videos on Technology, Entertainment and Design — plus science, business, global issues, the arts and more. Visit http://TED.com to get our entire library of TED Talks, transcripts, translations, personalized talk recommendations and more.\n\nWatch more: go.ted.com/billgates\n\nhttps://youtu.be/B5smctuV7-Q\n\nTED's videos may be used for non-commercial purposes under a Creative Commons License, Attribution–Non Commercial–No Derivatives (or the CC BY – NC – ND 4.0 International) and in accordance with our TED Talks Usage Policy (https://www.ted.com/about/our-organiz...). For more information on using TED for commercial purposes (e.g. employee learning, in a film or online course), please submit a Media Request at https://media-requests.ted.com"
1,"""Every great and difficult thing has required a strong sense of optimism,"" says editor and author Kevin Kelly, who believes that we have a moral obligation to be optimistic. Tracing humanity's progress throughout history, he's observed that a positive outlook helps us solve problems and empowers us to forge a path forward. In this illuminating talk, he shares three reasons for optimism during challenging times, explaining how it can help us become better ancestors and create the world we want to see for ourselves and future generations.\n\nIf you love watching TED Talks like this one, become a TED Member to support our mission of spreading ideas: http://ted.com/membership\n\nFollow TED! \nTwitter: http://twitter.com/TEDTalks\nInstagram: https://www.instagram.com/ted\nFacebook: http://facebook.com/TED\nLinkedIn: https://www.linkedin.com/company/ted-...\nTikTok: https://www.tiktok.com/@tedtoks\n\nThe TED Talks channel features talks, performances and original series from the world's leading thinkers and doers. Subscribe to our channel for videos on Technology, Entertainment and Design — plus science, business, global issues, the arts and more. Visit http://TED.com to get our entire library of TED Talks, transcripts, translations, personalized talk recommendations and more.\n\nWatch more: https://go.ted.com/kevinkelly\nhttps://youtu.be/FrqBWQ-mVEc\n\nTED's videos may be used for non-commercial purposes under a Creative Commons License, Attribution–Non Commercial–No Derivatives (or the CC BY – NC – ND 4.0 International) and in accordance with our TED Talks Usage Policy (https://www.ted.com/about/our-organiz...). For more information on using TED for commercial purposes (e.g. employee learning, in a film or online course), please submit a Media Request at https://media-requests.ted.com"
2,"Getting pregnant as a track and field athlete is often called the ""kiss of death"" -- a sign your athletic career will soon end. Olympic champion, entrepreneur and proud mother Allyson Felix thinks it shouldn't be that way. She tells the story of starting a family while fighting to change he

### Clean description column (many steps)

In [9]:
# Remove newline characters ('\r', '\n')
df['description'] = df['description'].str.replace('\r', ' ').str.replace('\n', ' ').astype(str)
df['description'].head()

,description
0,"Building a pandemic-free future won’t be easy, but Bill Gates believes that we have the tools and strategies to make it possible -- now we just have to fund them. In this forward-looking talk, he proposes a multi-specialty Global Epidemic Response and Mobilization (GERM) team that would detect potential outbreaks and stop them from becoming pandemics. By investing in disease monitoring, research and development as well as improved health systems, Gates believes we can “create a world where everyone has a chance to live a healthy and productive life -- a life free from the fear of the next COVID-19.” If you love watching TED Talks like this one, become a TED Member to support our mission of spreading ideas: http://ted.com/membership Follow TED! Twitter: http://twitter.com/TEDTalks Instagram: https://www.instagram.com/ted Facebook: http://facebook.com/TED LinkedIn: https://www.linkedin.com/company/ted-... TikTok: https://www.tiktok.com/@tedtoks The TED Talks channel features talks, performances and original series from the world's leading thinkers and doers. Subscribe to our channel for videos on Technology, Entertainment and Design — plus science, business, global issues, the arts and more. Visit http://TED.com to get our entire library of TED Talks, transcripts, translations, personalized talk recommendations and more. Watch more: go.ted.com/billgates https://youtu.be/B5smctuV7-Q TED's videos may be used for non-commercial purposes under a Creative Commons License, Attribution–Non Commercial–No Derivatives (or the CC BY – NC – ND 4.0 International) and in accordance with our TED Talks Usage Policy (https://www.ted.com/about/our-organiz...). For more information on using TED for commercial purposes (e.g. employee learning, in a film or online course), please submit a Media Request at https://media-requests.ted.com"
1,"""Every great and difficult thing has required a strong sense of optimism,"" says editor and author Kevin Kelly, who believes that we have a moral obligation to be optimistic. Tracing humanity's progress throughout history, he's observed that a positive outlook helps us solve problems and empowers us to forge a path forward. In this illuminating talk, he shares three reasons for optimism during challenging times, explaining how it can help us become better ancestors and create the world we want to see for ourselves and future generations. If you love watching TED Talks like this one, become a TED Member to support our mission of spreading ideas: http://ted.com/membership Follow TED! Twitter: http://twitter.com/TEDTalks Instagram: https://www.instagram.com/ted Facebook: http://facebook.com/TED LinkedIn: https://www.linkedin.com/company/ted-... TikTok: https://www.tiktok.com/@tedtoks The TED Talks channel features talks, performances and original series from the world's leading thinkers and doers. Subscribe to our channel for videos on Technology, Entertainment and Design — plus science, business, global issues, the arts and more. Visit http://TED.com to get our entire library of TED Talks, transcripts, translations, personalized talk recommendations and more. Watch more: https://go.ted.com/kevinkelly https://youtu.be/FrqBWQ-mVEc TED's videos may be used for non-commercial purposes under a Creative Commons License, Attribution–Non Commercial–No Derivatives (or the CC BY – NC – ND 4.0 International) and in accordance with our TED Talks Usage Policy (https://www.ted.com/about/our-organiz...). For more information on using TED for commercial purposes (e.g. employee learning, in a film or online course), please submit a Media Request at https://media-requests.ted.com"
2,"Getting pregnant as a track and field athlete is often called the ""kiss of death"" -- a sign your athletic career will soon end. Olympic champion, entrepreneur and proud mother Allyson Felix thinks it shouldn't be that way. She tells the story of starting a family while fighting to change her former sponsor's maternity policy -- and pav

In [10]:
# Remove rest of string when this string is encoutered:
#  'If you love watching TED Talks like this one, become a TED Member to support our mission of spreading ideas'
df['description'] = df['description'].str.replace(r'If you love watching TED Talks like this one, become a TED Member to support our mission of spreading ideas.*', "", case=False, regex=True)
df['description'].head(20)

,description
0,"Building a pandemic-free future won’t be easy, but Bill Gates believes that we have the tools and strategies to make it possible -- now we just have to fund them. In this forward-looking talk, he proposes a multi-specialty Global Epidemic Response and Mobilization (GERM) team that would detect potential outbreaks and stop them from becoming pandemics. By investing in disease monitoring, research and development as well as improved health systems, Gates believes we can “create a world where everyone has a chance to live a healthy and productive life -- a life free from the fear of the next COVID-19.”"
1,"""Every great and difficult thing has required a strong sense of optimism,"" says editor and author Kevin Kelly, who believes that we have a moral obligation to be optimistic. Tracing humanity's progress throughout history, he's observed that a positive outlook helps us solve problems and empowers us to forge a path forward. In this illuminating talk, he shares three reasons for optimism during challenging times, explaining how it can help us become better ancestors and create the world we want to see for ourselves and future generations."
2,"Getting pregnant as a track and field athlete is often called the ""kiss of death"" -- a sign your athletic career will soon end. Olympic champion, entrepreneur and proud mother Allyson Felix thinks it shouldn't be that way. She tells the story of starting a family while fighting to change her former sponsor's maternity policy -- and paving the way for others to get greater protection and more support. Her message is a testament to the power of believing in and advocating for yourself. “You don’t have to be an Olympian to create change for yourself and others,"" she says. ""Each of us can bet on ourselves."""
3,"The peatlands of Africa's Congo Basin are a vast expanse of swamp and greenery that act as one of the world's most effective carbon sinks -- and they're under threat of environmental destruction. Economist Vera Songwe explains how putting a price on the carbon stored in the peatlands would not only help protect this vital resource but also recognize and reward the African communities that have contributed little to climate change. ""This is not just about decarbonization,"" Songwe says. ""This is also about development with dignity."" Countdown is TED's global initiative to accelerate solutions to the climate crisis. The goal: to build a better future by cutting greenhouse gas emissions in half by 2030, in the race to a zero-carbon world. Get involved at https://countdown.ted.com/sign-up Learn more about #TEDCountdown: Twitter: https://twitter.com/TEDCountdown Instagram: https://www.instagram.com/tedcountdown Facebook: https://www.facebook.com/TED Website: https://countdown.ted.com Watch the full 2021 TED Countdown Global livestream here: https://youtu.be/SG_vqlb1pOQ"
4,"What's on Elon Musk's mind? In this exclusive conversation with head of TED Chris Anderson, Musk details how the radical new innovations he's working on -- Tesla's intelligent humanoid robot Optimus, SpaceX's otherworldly Starship and Neuralink's brain-machine interfaces, among others -- could help maximize the lifespan of humanity and create a world where goods and services are abundant and accessible for all. It's a compelling vision of a future worth getting excited about. (Recorded at the Tesla Texas Gigafactory on April 6, 2022) Just over a week after this interview was filmed, Elon Musk joined TED2022 for another (live) conversation, where he discussed his bid to purchase Twitter, the biggest regret of his career, how his brain works and more. Watch that conversation here: https://youtu.be/cdZZpaB2kDM 0:14 A future that's worth getting excited about 2:44 The sustainable energy economy, batteries and 300 terawatt hours of installed capacity 7:06 ""Humanity will solve sustainable energy."" 8:47 Artificial intelligence and Tesla's progress on full self-driving cars 19:46 Tesla's Optimus humanoid 

In [11]:
# Remove rest of string when this string is encoutered:
#  'Visit ted.com/membership to become a TED Member'
df['description'] = df['description'].str.replace(r'Visit ted.com/membership to become a TED Member*', "", case=False, regex=True)
df['description'].head(20)

,description
0,"Building a pandemic-free future won’t be easy, but Bill Gates believes that we have the tools and strategies to make it possible -- now we just have to fund them. In this forward-looking talk, he proposes a multi-specialty Global Epidemic Response and Mobilization (GERM) team that would detect potential outbreaks and stop them from becoming pandemics. By investing in disease monitoring, research and development as well as improved health systems, Gates believes we can “create a world where everyone has a chance to live a healthy and productive life -- a life free from the fear of the next COVID-19.”"
1,"""Every great and difficult thing has required a strong sense of optimism,"" says editor and author Kevin Kelly, who believes that we have a moral obligation to be optimistic. Tracing humanity's progress throughout history, he's observed that a positive outlook helps us solve problems and empowers us to forge a path forward. In this illuminating talk, he shares three reasons for optimism during challenging times, explaining how it can help us become better ancestors and create the world we want to see for ourselves and future generations."
2,"Getting pregnant as a track and field athlete is often called the ""kiss of death"" -- a sign your athletic career will soon end. Olympic champion, entrepreneur and proud mother Allyson Felix thinks it shouldn't be that way. She tells the story of starting a family while fighting to change her former sponsor's maternity policy -- and paving the way for others to get greater protection and more support. Her message is a testament to the power of believing in and advocating for yourself. “You don’t have to be an Olympian to create change for yourself and others,"" she says. ""Each of us can bet on ourselves."""
3,"The peatlands of Africa's Congo Basin are a vast expanse of swamp and greenery that act as one of the world's most effective carbon sinks -- and they're under threat of environmental destruction. Economist Vera Songwe explains how putting a price on the carbon stored in the peatlands would not only help protect this vital resource but also recognize and reward the African communities that have contributed little to climate change. ""This is not just about decarbonization,"" Songwe says. ""This is also about development with dignity."" Countdown is TED's global initiative to accelerate solutions to the climate crisis. The goal: to build a better future by cutting greenhouse gas emissions in half by 2030, in the race to a zero-carbon world. Get involved at https://countdown.ted.com/sign-up Learn more about #TEDCountdown: Twitter: https://twitter.com/TEDCountdown Instagram: https://www.instagram.com/tedcountdown Facebook: https://www.facebook.com/TED Website: https://countdown.ted.com Watch the full 2021 TED Countdown Global livestream here: https://youtu.be/SG_vqlb1pOQ"
4,"What's on Elon Musk's mind? In this exclusive conversation with head of TED Chris Anderson, Musk details how the radical new innovations he's working on -- Tesla's intelligent humanoid robot Optimus, SpaceX's otherworldly Starship and Neuralink's brain-machine interfaces, among others -- could help maximize the lifespan of humanity and create a world where goods and services are abundant and accessible for all. It's a compelling vision of a future worth getting excited about. (Recorded at the Tesla Texas Gigafactory on April 6, 2022) Just over a week after this interview was filmed, Elon Musk joined TED2022 for another (live) conversation, where he discussed his bid to purchase Twitter, the biggest regret of his career, how his brain works and more. Watch that conversation here: https://youtu.be/cdZZpaB2kDM 0:14 A future that's worth getting excited about 2:44 The sustainable energy economy, batteries and 300 terawatt hours of installed capacity 7:06 ""Humanity will solve sustainable energy."" 8:47 Artificial intelligence and Tesla's progress on full self-driving cars 19:46 Tesla's Optimus humanoid 

In [12]:
# Remove URLs (both http and https)
df['description'] = df['description'].str.replace(r'http[s]?://\S+', "", regex=True)
df['description'].head(20)

,description
0,"Building a pandemic-free future won’t be easy, but Bill Gates believes that we have the tools and strategies to make it possible -- now we just have to fund them. In this forward-looking talk, he proposes a multi-specialty Global Epidemic Response and Mobilization (GERM) team that would detect potential outbreaks and stop them from becoming pandemics. By investing in disease monitoring, research and development as well as improved health systems, Gates believes we can “create a world where everyone has a chance to live a healthy and productive life -- a life free from the fear of the next COVID-19.”"
1,"""Every great and difficult thing has required a strong sense of optimism,"" says editor and author Kevin Kelly, who believes that we have a moral obligation to be optimistic. Tracing humanity's progress throughout history, he's observed that a positive outlook helps us solve problems and empowers us to forge a path forward. In this illuminating talk, he shares three reasons for optimism during challenging times, explaining how it can help us become better ancestors and create the world we want to see for ourselves and future generations."
2,"Getting pregnant as a track and field athlete is often called the ""kiss of death"" -- a sign your athletic career will soon end. Olympic champion, entrepreneur and proud mother Allyson Felix thinks it shouldn't be that way. She tells the story of starting a family while fighting to change her former sponsor's maternity policy -- and paving the way for others to get greater protection and more support. Her message is a testament to the power of believing in and advocating for yourself. “You don’t have to be an Olympian to create change for yourself and others,"" she says. ""Each of us can bet on ourselves."""
3,"The peatlands of Africa's Congo Basin are a vast expanse of swamp and greenery that act as one of the world's most effective carbon sinks -- and they're under threat of environmental destruction. Economist Vera Songwe explains how putting a price on the carbon stored in the peatlands would not only help protect this vital resource but also recognize and reward the African communities that have contributed little to climate change. ""This is not just about decarbonization,"" Songwe says. ""This is also about development with dignity."" Countdown is TED's global initiative to accelerate solutions to the climate crisis. The goal: to build a better future by cutting greenhouse gas emissions in half by 2030, in the race to a zero-carbon world. Get involved at Learn more about #TEDCountdown: Twitter: Instagram: Facebook: Website: Watch the full 2021 TED Countdown Global livestream here:"
4,"What's on Elon Musk's mind? In this exclusive conversation with head of TED Chris Anderson, Musk details how the radical new innovations he's working on -- Tesla's intelligent humanoid robot Optimus, SpaceX's otherworldly Starship and Neuralink's brain-machine interfaces, among others -- could help maximize the lifespan of humanity and create a world where goods and services are abundant and accessible for all. It's a compelling vision of a future worth getting excited about. (Recorded at the Tesla Texas Gigafactory on April 6, 2022) Just over a week after this interview was filmed, Elon Musk joined TED2022 for another (live) conversation, where he discussed his bid to purchase Twitter, the biggest regret of his career, how his brain works and more. Watch that conversation here: 0:14 A future that's worth getting excited about 2:44 The sustainable energy economy, batteries and 300 terawatt hours of installed capacity 7:06 ""Humanity will solve sustainable energy."" 8:47 Artificial intelligence and Tesla's progress on full self-driving cars 19:46 Tesla's Optimus humanoid robot 21:46 ""People have no idea, this is going to be bigger than the car."" 23:14 Avoiding an AI dystopia 26:39 The age of abundance 28:20 Neuralink and brain-machine interfaces 36:55 SpaceX's Starship and the mission

In [13]:
# Remove common words / website domains:
#  'TEDCountdown', 'Twitter', 'Instagram', 'Facebook', 'Website', 'linkedin', 'tiktok'
words_to_remove = ['TEDCountdown', 'Twitter', 'Instagram', 'Facebook', 'Website', 'linkedin', 'tiktok']
pattern = '|'.join(words_to_remove)
df['description'] = df['description'].str.replace(pattern, "", case=False, regex=True)
df['description'].head(20)

,description
0,"Building a pandemic-free future won’t be easy, but Bill Gates believes that we have the tools and strategies to make it possible -- now we just have to fund them. In this forward-looking talk, he proposes a multi-specialty Global Epidemic Response and Mobilization (GERM) team that would detect potential outbreaks and stop them from becoming pandemics. By investing in disease monitoring, research and development as well as improved health systems, Gates believes we can “create a world where everyone has a chance to live a healthy and productive life -- a life free from the fear of the next COVID-19.”"
1,"""Every great and difficult thing has required a strong sense of optimism,"" says editor and author Kevin Kelly, who believes that we have a moral obligation to be optimistic. Tracing humanity's progress throughout history, he's observed that a positive outlook helps us solve problems and empowers us to forge a path forward. In this illuminating talk, he shares three reasons for optimism during challenging times, explaining how it can help us become better ancestors and create the world we want to see for ourselves and future generations."
2,"Getting pregnant as a track and field athlete is often called the ""kiss of death"" -- a sign your athletic career will soon end. Olympic champion, entrepreneur and proud mother Allyson Felix thinks it shouldn't be that way. She tells the story of starting a family while fighting to change her former sponsor's maternity policy -- and paving the way for others to get greater protection and more support. Her message is a testament to the power of believing in and advocating for yourself. “You don’t have to be an Olympian to create change for yourself and others,"" she says. ""Each of us can bet on ourselves."""
3,"The peatlands of Africa's Congo Basin are a vast expanse of swamp and greenery that act as one of the world's most effective carbon sinks -- and they're under threat of environmental destruction. Economist Vera Songwe explains how putting a price on the carbon stored in the peatlands would not only help protect this vital resource but also recognize and reward the African communities that have contributed little to climate change. ""This is not just about decarbonization,"" Songwe says. ""This is also about development with dignity."" Countdown is TED's global initiative to accelerate solutions to the climate crisis. The goal: to build a better future by cutting greenhouse gas emissions in half by 2030, in the race to a zero-carbon world. Get involved at Learn more about #: : : : : Watch the full 2021 TED Countdown Global livestream here:"
4,"What's on Elon Musk's mind? In this exclusive conversation with head of TED Chris Anderson, Musk details how the radical new innovations he's working on -- Tesla's intelligent humanoid robot Optimus, SpaceX's otherworldly Starship and Neuralink's brain-machine interfaces, among others -- could help maximize the lifespan of humanity and create a world where goods and services are abundant and accessible for all. It's a compelling vision of a future worth getting excited about. (Recorded at the Tesla Texas Gigafactory on April 6, 2022) Just over a week after this interview was filmed, Elon Musk joined TED2022 for another (live) conversation, where he discussed his bid to purchase , the biggest regret of his career, how his brain works and more. Watch that conversation here: 0:14 A future that's worth getting excited about 2:44 The sustainable energy economy, batteries and 300 terawatt hours of installed capacity 7:06 ""Humanity will solve sustainable energy."" 8:47 Artificial intelligence and Tesla's progress on full self-driving cars 19:46 Tesla's Optimus humanoid robot 21:46 ""People have no idea, this is going to be bigger than the car."" 23:14 Avoiding an AI dystopia 26:39 The age of abundance 28:20 Neuralink and brain-machine interfaces 36:55 SpaceX's Starship and the mission to build a city on Mars 46:54 ""It's the people o

### Create function to preprocess text

In [14]:
# Function to clean_text
def clean_text(text):
    lem = WordNetLemmatizer()
    stop = set(stopwords.words('english'))
    punct = string.punctuation
    text = re.sub(r'\s+', ' ', text)
    text = text.translate(str.maketrans('', '', punct)).lower()
    tokens = re.split(r'\W+', text)
    tokens = [lem.lemmatize(word) for word in tokens if word not in stop]
    return ' '.join(tokens)

### Use function to clean column 'description'

In [15]:
%%time

df['description_clean'] = df['description'].apply(clean_text)
df['description_clean'].head()

CPU times: user 6.4 s, sys: 255 ms, total: 6.65 s
Wall time: 11 s


,description_clean
0,building pandemicfree future easy bill gate belief tool strategy make possible fund forwardlooking talk proposes multispecialty global epidemic response mobilization germ team would detect potential outbreak stop becoming pandemic investing disease monitoring research development well improved health system gate belief create world everyone chance live healthy productive life life free fear next covid19
1,every great difficult thing required strong sense optimism say editor author kevin kelly belief moral obligation optimistic tracing humanity progress throughout history he observed positive outlook help u solve problem empowers u forge path forward illuminating talk share three reason optimism challenging time explaining help u become better ancestor create world want see future generation
2,getting pregnant track field athlete often called kiss death sign athletic career soon end olympic champion entrepreneur proud mother allyson felix think shouldnt way tell story starting family fighting change former sponsor maternity policy paving way others get greater protection support message testament power believing advocating olympian create change others say u bet
3,peatlands africa congo basin vast expanse swamp greenery act one world effective carbon sink theyre threat environmental destruction economist vera songwe explains putting price carbon stored peatlands would help protect vital resource also recognize reward african community contributed little climate change decarbonization songwe say also development dignity countdown ted global initiative accelerate solution climate crisis goal build better future cutting greenhouse gas emission half 2030 race zerocarbon world get involved learn watch full 2021 ted countdown global livestream
4,whats elon musk mind exclusive conversation head ted chris anderson musk detail radical new innovation he working tesla intelligent humanoid robot optimus spacexs otherworldly starship neuralinks brainmachine interface among others could help maximize lifespan humanity create world good service abundant accessible compelling vision future worth getting excited recorded tesla texas gigafactory april 6 2022 week interview filmed elon musk joined ted2022 another live conversation discussed bid purchase biggest regret career brain work watch conversation 014 future thats worth getting excited 244 sustainable energy economy battery 300 terawatt hour installed capacity 706 humanity solve sustainable energy 847 artificial intelligence tesla progress full selfdriving car 1946 tesla optimus humanoid robot 2146 people idea going bigger car 2314 avoiding ai dystopia 2639 age abundance 2820 neuralink brainmachine interface 3655 spacexs starship mission build city mar 4654 people mar city 5014 else starship help explore 5318 possible synergy tesla spacex boring company neuralink 5444 intercontinental travel via starship 5841 billionaire 10231 philanthropy love humanity 10339 population collapse birth rate threat future human civilization 10413 elons drive 10606 think want future good must make


### Split data into training and validation sets
* Reserve 0.005% of data to test with later

In [16]:
df1, df2 = train_test_split(df, test_size=.005, random_state=42)

In [17]:
df1.shape

(3903, 10)

In [18]:
df2.shape

(20, 10)

## Part 1: Shingle is determined by word boundary

### Create function to generate MinHash Forest
* Initialize number of permutations in MinHash
* MinHash the string on all shingles in each document
* Store the MinHash of the string
* Generate a forest of all MinHashed strings
* Index the forest to make it searchable

In [19]:
def generate_forest(docs, permutations):
    start_time = time.time()

    minhash = []

    for doc in docs:
        m = MinHash(num_perm=permutations)
        for token in doc:                      # Process shingles on word boundary
            m.update(token.encode('utf8'))
        minhash.append(m)

    forest = MinHashLSHForest(num_perm=permutations)

    for i,m in enumerate(minhash):
        forest.add(i,m)

    forest.index()

    print('It took %s seconds to build forest.' %(time.time()-start_time))

    return forest

### Create function to query MinHash Forest
* Preprocess input text into shingles
* Use the same number of permutations for the MinHash as was used to build the forest
* Create a MinHash on the input text using all shingles
* Query the forest with MinHash and return the number of requested recommendations
* Provide the titles of each conference paper recommended

In [20]:
def predict(text, df, permutations, num_results, forest):
    start_time = time.time()

    m = MinHash(num_perm=permutations)
    for token in text:
        m.update(token.encode('utf8'))

    idx_array = np.array(forest.query(m, num_results))
    if len(idx_array) == 0:
        return None     # if query is empty, return none

    result = df.iloc[idx_array]['title']

    print('It took %s seconds to query forest.' %(time.time()-start_time))

    return result

### Create forest

In [21]:
# Set number of Permutations
permutations = 128

In [22]:
forest = generate_forest(df1['description_clean'], permutations)

It took 36.25963759422302 seconds to build forest.


### Use forest to make recommendations for 2 different TED talks

In [23]:
idx = 1
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.013104677200317383 seconds to query forest.

Top 5 recommendations for [You shouldn't have to choose between filling your prescriptions and paying bills | Kiah Williams]:
2462                                                           Jake Barton: The museum of you
410                        Why children stay silent following sexual violence | Kristin Jones
495                           A new stock exchange focused on the long-term | Michelle Greene
432     What if a US presidential candidate refuses to concede after an election? | Van Jones
70                      A King Cobra Bite -- and a Scientific Discovery | Gowri Shankar | TED
Name: title, dtype: object


In [24]:
idx = 2
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.009930610656738281 seconds to query forest.

Top 5 recommendations for [John Hockenberry: We are all designers]:
1964                       Street Art for Hope and Peace | eL Seed | TED Talks
2897                                        Robin Ince: Science versus wonder?
555     How racial bias works -- and how to disrupt it | Jennifer L. Eberhardt
2181         Humble plants that hide surprising secrets | Ameenah Gurib-Fakim:
2980                           Jonathan Drori: The beautiful tricks of flowers
Name: title, dtype: object


## Part 2: Shingle size is fixed

### Create function to create shingles

In [25]:
def create_shingles(text, shingle_size=6):
    return [text[i:i+shingle_size] for i in range(len(text)-shingle_size+1)]

### Create function to generate MinHash Forest (using fixed shingle size)
* Initialize number of permutations in MinHash
* MinHash the string on all shingles in each document
* Store the MinHash of the string
* Generate a forest of all MinHashed strings
* Index the forest to make it searchable

In [26]:
def generate_forest_2(docs, permutations):
    start_time = time.time()

    minhash = []

    for doc in docs:
        shingles = create_shingles(doc)
        m = MinHash(num_perm=permutations)
        for shingle in shingles:
            m.update(shingle.encode('utf8'))
        minhash.append(m)

    forest = MinHashLSHForest(num_perm=permutations)

    for i,m in enumerate(minhash):
        forest.add(i,m)

    forest.index()

    print('It took %s seconds to build forest.' %(time.time()-start_time))

    return forest

### Create function to query MinHash Forest (using fixed shingle size)
* Preprocess input text into fixed size shingles
* Use the same number of permutations for the MinHash as was used to build the forest
* Create a MinHash on the input text using all shingles
* Query the forest with MinHash and return the number of requested recommendations
* Provide the titles of each conference paper recommended

In [27]:
def predict_2(text, df, permutations, num_results, forest):
    start_time = time.time()

    tokens = clean_text(text)
    shingles = create_shingles(tokens)
    m = MinHash(num_perm=permutations)
    for shingle in shingles:
        m.update(shingle.encode('utf8'))

    idx_array = np.array(forest.query(m, num_results))
    if len(idx_array) == 0:
        return None     # if query is empty, return none

    result = df.iloc[idx_array]['title']

    print('It took %s seconds to query forest.' %(time.time()-start_time))

    return result

### Create forest (using fixed shingle size)

In [28]:
forest = generate_forest_2(df1['description_clean'], permutations)

It took 27.202327966690063 seconds to build forest.


### Use forest (with fixed shingle size) to make recommendations for 2 different TED talks

In [29]:
idx = 1
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict_2(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.012094497680664062 seconds to query forest.

Top 5 recommendations for [You shouldn't have to choose between filling your prescriptions and paying bills | Kiah Williams]:
264     Matthew Mazzotta: Playful, wondrous public spaces built for community and possibility | TED Fellows
261                            Why you don't need 8 glasses of water a day | Body Stuff with Dr. Jen Gunter
2667                                                            A 12-year-old app developer | Thomas Suarez
233           Sahaj Kaur Kohli: Why children of immigrants experience guilt -- and strategies to cope | TED
161                                                                The 55 Gigaton Challenge | TED Countdown
Name: title, dtype: object


In [30]:
idx = 2
num_recommendations = 5
input_title = df2.iloc[idx]['title']
input_text = df2.iloc[idx]['description_clean']
results = predict_2(input_text, df1, permutations, num_recommendations, forest)
print(f'\nTop {num_recommendations} recommendations for [{input_title}]:')
print(results)

It took 0.01210641860961914 seconds to query forest.

Top 5 recommendations for [John Hockenberry: We are all designers]:
2764                         John Hodgman: Design, explained.
3205    Jason Clay: How big brands can help save biodiversity
3536                   Nalini Nadkarni explores canopy worlds
3286           Jonathan Drori: Every pollen grain has a story
3000              Paul Romer: The world's first charter city?
Name: title, dtype: object
